In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import subprocess
subprocess.run(["pip", "install", "timm", "-q"], check=True)
print("timm installed!")

In [ ]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
 
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"timm   : {timm.__version__}")

In [ ]:
ROOT       = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1"
IMG_DIR    = f"{ROOT}/images"
TRAIN_CSV  = f"{ROOT}/train.csv"
TEST_CSV   = f"{ROOT}/test.csv"
SAMPLE_SUB = f"{ROOT}/sample_submission.csv"
CKPT       = "/kaggle/working/best_swin.pth"
SUB_PATH   = "/kaggle/working/submission.csv"
 
IMG_SIZE   = 224     # Swin-T native size
BATCH      = 16      # smaller batch for transformer
N_EPOCHS   = 25
LR         = 5e-5    # transformers need lower LR
N_TTA      = 7
PATIENCE   = 6

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
 
CLASS_COLS  = [c for c in train_df.columns if c != "id"]
N_CLASSES   = len(CLASS_COLS)
NO_FIND_IDX = CLASS_COLS.index("No Finding") if "No Finding" in CLASS_COLS else 0
 
print(f"Classes : {N_CLASSES}")
print(f"No Find : {NO_FIND_IDX}")
 
# One-hot → single class index
oh       = train_df[CLASS_COLS].values.astype(np.int32)
row_sums = oh.sum(axis=1)
targets  = np.where(row_sums > 0, oh.argmax(axis=1), NO_FIND_IDX)
train_df["target"] = targets
 
counts = np.bincount(targets, minlength=N_CLASSES).astype(np.float32)
print(f"\nClass distribution:")
for i, (col, cnt) in enumerate(zip(CLASS_COLS, counts)):
    print(f"  [{i:2d}] {col:<35} n={int(cnt):>6}")
 
# Stratified 90/10 split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, vl_idx = next(sss.split(train_df, targets))
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
vl_df = train_df.iloc[vl_idx].reset_index(drop=True)
print(f"\nTrain: {len(tr_df)} | Val: {len(vl_df)}")

In [ ]:
class CXRDataset(Dataset):
    def __init__(self, df, img_dir, tfm, is_test=False):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfm     = tfm
        self.is_test = is_test
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, i):
        row = self.df.loc[i]
        img = Image.open(
            os.path.join(self.img_dir, row["id"])
        ).convert("RGB")
        img = self.tfm(img)
        if self.is_test:
            return img, row["id"]
        return img, int(row["target"])

In [ ]:
mu, sd = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
 
tr_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mu, sd),
])
 
ev_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mu, sd),
])
 
tta_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize(mu, sd),
])
 

In [ ]:
sample_w = np.array(
    [1.0 / counts[t] for t in tr_df["target"]],
    dtype=np.float32
)
sampler = WeightedRandomSampler(
    torch.from_numpy(sample_w), len(tr_df), replacement=True
)
 
tr_ds  = CXRDataset(tr_df,  IMG_DIR, tr_tfm)
vl_ds  = CXRDataset(vl_df,  IMG_DIR, ev_tfm)
ts_ds  = CXRDataset(test_df, IMG_DIR, ev_tfm, is_test=True)
 
tr_ld = DataLoader(tr_ds, batch_size=BATCH, sampler=sampler,
                   num_workers=2, pin_memory=True)
vl_ld = DataLoader(vl_ds, batch_size=BATCH, shuffle=False,
                   num_workers=2, pin_memory=True)
ts_ld = DataLoader(ts_ds, batch_size=BATCH, shuffle=False,
                   num_workers=2, pin_memory=True)

In [ ]:
net = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained  = True,
    num_classes = N_CLASSES,
)
net = net.to(DEVICE)
 
total  = sum(p.numel() for p in net.parameters())
print(f"\nModel : Swin-T (swin_tiny_patch4_window7_224)")
print(f"Params: {total:,}")
 
# Verify forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = net(dummy)
    print(f"Output shape: {out.shape}")  # should be [2, N_CLASSES]

In [ ]:
cls_w   = counts.sum() / (N_CLASSES * counts + 1e-6)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)
 
criterion = nn.CrossEntropyLoss(weight=cls_w_t)
 
# AdamW is standard for transformers
optimizer = torch.optim.AdamW(
    net.parameters(),
    lr           = LR,
    weight_decay = 0.05,   # standard for Swin
)
 
# Cosine annealing with warmup
def warmup_cosine(epoch):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (N_EPOCHS - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))
 
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=warmup_cosine
)

In [ ]:
def comp_score(y_true, y_pred):
    """Score_c = (TP - FP - 5*FN) / Nc, averaged over classes."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []
    for c in range(N_CLASSES):
        Nc = (y_true == c).sum()
        if Nc == 0:
            scores.append(0.0); continue
        TP = ((y_true == c) & (y_pred == c)).sum()
        FP = ((y_true != c) & (y_pred == c)).sum()
        FN = ((y_true == c) & (y_pred != c)).sum()
        scores.append((TP - FP - 5 * FN) / Nc)
    return float(np.mean(scores))

In [ ]:
def train_epoch(model, loader, opt, crit):
    model.train()
    total = 0.0
    for imgs, labels in tqdm(loader, desc="  train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(imgs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)
 
 
def validate(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
            imgs = imgs.to(DEVICE)
            all_p.extend(model(imgs).argmax(1).cpu().numpy())
            all_l.extend(labels.numpy())
    all_p = np.array(all_p)
    all_l = np.array(all_l)
    f1 = f1_score(all_l, all_p, average="macro", zero_division=0)
    cs = comp_score(all_l, all_p)
    return f1, cs
 

In [ ]:
best_score = -9999
no_improve = 0
 
print("\n" + "="*65)
print("Swin-T | IMG=224 | BATCH=16 | LR=5e-5 | Epochs=25")
print("="*65)
 
for ep in range(1, N_EPOCHS + 1):
    tr_loss  = train_epoch(net, tr_ld, optimizer, criterion)
    vf1, vcs = validate(net, vl_ld)
    scheduler.step()
 
    cur_lr = optimizer.param_groups[0]["lr"]
    flag   = ""
 
    if vcs > best_score:
        best_score = vcs
        no_improve = 0
        torch.save(net.state_dict(), CKPT)
        flag = "saved"
    else:
        no_improve += 1
        flag = f"  ({no_improve}/{PATIENCE})"
 
    print(f"Ep {ep:02d}/{N_EPOCHS} | "
          f"loss={tr_loss:.4f} | "
          f"F1={vf1:.4f} | "
          f"CompScore={vcs:.4f} | "
          f"lr={cur_lr:.2e}"
          f"{flag}")
 
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {ep}")
        break
 
print(f"\nBest CompScore: {best_score:.4f}")

In [ ]:
print("\nPer-class scale tuning...")
net.load_state_dict(torch.load(CKPT, map_location=DEVICE))
net.eval()
 
val_probs, val_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(vl_ld, desc="  probs", leave=False):
        p = F.softmax(net(imgs.to(DEVICE)), dim=1)
        val_probs.append(p.cpu().numpy())
        val_labels.extend(labels.numpy())
 
val_probs  = np.vstack(val_probs)
val_labels = np.array(val_labels)
 
baseline = comp_score(val_labels, val_probs.argmax(1))
print(f"Baseline: {baseline:.4f}")
 
best_scales = np.ones(N_CLASSES, dtype=np.float32)
for c in range(N_CLASSES):
    best_s  = 1.0
    best_sc = baseline
    for s in [1.3, 1.6, 2.0, 2.5, 3.0]:
        trial = val_probs.copy()
        trial[:, c] *= s
        sc = comp_score(val_labels, trial.argmax(1))
        if sc > best_sc:
            best_sc = sc
            best_s  = s
    if best_s > 1.0:
        best_scales[c] = best_s
        print(f"  [{c:2d}] {CLASS_COLS[c]:<30} "
              f"scale={best_s:.1f} → {best_sc:.4f}")
 
tuned = val_probs.copy()
for c in range(N_CLASSES):
    tuned[:, c] *= best_scales[c]
tuned_score = comp_score(val_labels, tuned.argmax(1))
print(f"After tuning : {tuned_score:.4f}")
print(f"Improvement  : {tuned_score - baseline:+.4f}")
 
if tuned_score <= baseline:
    print("Tuning did not help — using raw probs")
    best_scales = np.ones(N_CLASSES, dtype=np.float32)

In [ ]:
print(f"\nTTA Inference ({N_TTA} passes)...")
 
ts_tta    = CXRDataset(test_df, IMG_DIR, tta_tfm, is_test=True)
ts_tta_ld = DataLoader(ts_tta, batch_size=BATCH,
                       shuffle=False, num_workers=2)
 
net.eval()
n_test    = len(test_df)
sum_probs = np.zeros((n_test, N_CLASSES), dtype=np.float32)
all_ids   = []
 
# Clean pass
with torch.no_grad():
    offset = 0
    for imgs, fnames in tqdm(ts_ld, desc="  clean", leave=False):
        p = F.softmax(net(imgs.to(DEVICE)), dim=1).cpu().numpy()
        sum_probs[offset:offset+len(imgs)] += p
        all_ids.extend(fnames)
        offset += len(imgs)
 
# TTA passes
for t in range(N_TTA - 1):
    with torch.no_grad():
        offset = 0
        for imgs, _ in tqdm(ts_tta_ld,
                            desc=f"  TTA {t+2}/{N_TTA}",
                            leave=False):
            p = F.softmax(net(imgs.to(DEVICE)), dim=1).cpu().numpy()
            sum_probs[offset:offset+len(imgs)] += p
            offset += len(imgs)
 
# Average + apply per-class scales
avg_probs = sum_probs / N_TTA
for c in range(N_CLASSES):
    avg_probs[:, c] *= best_scales[c]
 
final_preds = avg_probs.argmax(axis=1)

In [ ]:
records = []
for fname, pred in zip(all_ids, final_preds):
    row = {"id": fname}
    for j, col in enumerate(CLASS_COLS):
        row[col] = int(pred == j)
    records.append(row)
 
sub    = pd.DataFrame(records)
sample = pd.read_csv(SAMPLE_SUB)
sub    = sub[sample.columns]
 
# Validate one-hot format
arr = sub[CLASS_COLS].to_numpy()
assert np.isin(arr, [0, 1]).all(),      "Values must be 0 or 1!"
assert (arr.sum(axis=1) == 1).all(),    "Each row must have exactly one 1!"
 
sub.to_csv(SUB_PATH, index=False)
print(f"\nsubmission.csv saved!")
print(f"  Shape : {sub.shape}")
print(sub.head())
 
# Prediction distribution
dist = pd.Series(final_preds).map(
    lambda x: CLASS_COLS[x]
).value_counts()
print(f"\nPrediction distribution:\n{dist}")